# L3 – Ejercicio: Elementos básicos de Spark (RDDs)

En este notebook vamos a construir, paso a paso, un flujo de procesamiento en **PySpark** usando RDDs, tal como pide el ejercicio de la lección 3.

## 1. Inicialización de SparkContext

1. Instalamos PySpark si estamos en Google Colab.
2. Creamos el `SparkContext` en modo local, usando todos los núcleos disponibles.
3. Bajamos el nivel de logs para que la salida sea más legible.

In [1]:
# Paso 1: instalación (solo necesaria en Colab; en local puedes omitirla)
# !pip -q install pyspark

from pyspark import SparkContext

sc = SparkContext(master='local[*]', appName='L3-RDD-Basics')
sc.setLogLevel('WARN')
sc

<SparkContext master=local[*] appName=L3-RDD-Basics>

## 2. Carga de datos en un RDD

1. Definimos la ruta de un archivo `.txt` (puede ser cualquier texto grande: Shakespeare, logs, etc.).
2. Usamos `sc.textFile(ruta)` para crear un RDD donde **cada elemento es una línea**.
3. Contamos cuántas líneas hay y mostramos las primeras para verificar la carga.

In [ ]:
# Paso 2: carga de datos
# Paso 2: carga de datos
ruta = r'D:\000_ANALISTA_DATOS\009_MODULO_9\EJERCICIOS\002_aplcacion_spark\shakespeare.txt'

text_rdd = sc.textFile(ruta)

print('Líneas:', text_rdd.count())
print('Primeras líneas:', text_rdd.take(3))
  # ajusta según tu entorno (ruta local o de Colab)

text_rdd = sc.textFile(ruta)

print('Líneas:', text_rdd.count())
print('Primeras líneas:', text_rdd.take(3))

## 3. Transformaciones fundamentales sobre el RDD

En este bloque aplicamos las transformaciones clave que pide el ejercicio:

1. `flatMap`: dividir cada línea en palabras.
2. `map`: normalizar las palabras (minúsculas, sin signos).
3. `filter`: descartar cadenas vacías o no válidas.
4. `distinct`: contar cuántas palabras únicas hay.
5. `map` a Pair-RDD `(palabra, 1)` y `reduceByKey` para frecuencias.
6. `sortBy`: ordenar por frecuencia descendente para obtener el Top 20.

In [ ]:
# Paso 3: transformaciones sobre el RDD de texto
import string

# flatMap: convertir líneas en palabras
words_rdd = (
    text_rdd
    .flatMap(lambda line: line.split())
    .map(lambda w: w.lower().strip(string.punctuation))
    .filter(lambda w: len(w) > 0)
)

# distinct: palabras únicas
unique_words = words_rdd.distinct().count()
print('Palabras únicas:', unique_words)

# Pair-RDD (palabra, 1) y frecuencias
pairs_rdd = words_rdd.map(lambda w: (w, 1))
freq_rdd = pairs_rdd.reduceByKey(lambda a, b: a + b)

# sortBy por frecuencia descendente y Top 20
top20 = freq_rdd.sortBy(lambda kv: kv[1], ascending=False).take(20)
print('Top 20 palabras:', top20)

## 4. Acciones típicas sobre RDD

Ahora ejecutamos distintas **acciones** para materializar el DAG y disparar Jobs en Spark:

1. `count()` sobre `words_rdd`.
2. `takeSample()` para ver una muestra.
3. Estadísticas numéricas (`sum`, `mean`, `stdev`) sobre las frecuencias.

In [ ]:
# Paso 4: acciones sobre el RDD de palabras y frecuencias
total_palabras = words_rdd.count()
muestra = words_rdd.takeSample(False, 10)

print('Total de palabras:', total_palabras)
print('Muestra de palabras:', muestra)

# Estadísticas sobre las frecuencias (valores del Pair-RDD)
freq_values_rdd = freq_rdd.map(lambda kv: kv[1])

print('Sum  :', freq_values_rdd.sum())
print('Mean :', freq_values_rdd.mean())
print('Stdev:', freq_values_rdd.stdev())

## 5. Linaje (`toDebugString`) y Lazy Evaluation

Con `toDebugString()` podemos inspeccionar el **linaje** (DAG) del RDD, es decir, la secuencia de
transformaciones que Spark tiene registradas. Estas transformaciones no se ejecutan hasta que
llamamos a una **acción** (`count`, `take`, `sum`, etc.). En ese momento, Spark:

1. Construye el plan de ejecución.
2. Lo divide en *stages* y *tasks*.
3. Lanza uno o más *Jobs* para obtener el resultado.

In [ ]:
# Paso 5: mostrar el linaje del RDD de frecuencias
print(freq_rdd.toDebugString())

## 6. Persistencia / caché y comparación de tiempos

El ejercicio pide **persistir** un RDD clave y comparar tiempos con y sin caché.

Pasos:
1. Persistimos `words_rdd` en memoria (`MEMORY_ONLY`).
2. Ejecutamos una acción para materializar el caché.
3. Medimos el tiempo de operaciones repetidas (por ejemplo `distinct` y `sample`).
4. (Opcional) Comparamos con la misma operación sin caché.

In [ ]:
# Paso 6: persistencia y tiempos
from pyspark import StorageLevel
import time

cached_words = words_rdd.persist(StorageLevel.MEMORY_ONLY)

# Primera acción: materializa el caché
_ = cached_words.count()

t0 = time.time(); _ = cached_words.distinct().count(); t1 = time.time()
t2 = time.time(); _ = cached_words.sample(False, 0.2).count(); t3 = time.time()

print(f'Distinct (cached)  : {t1 - t0:.3f}s')
print(f'Sample 20% (cached): {t3 - t2:.3f}s')

# (Opcional) limpiar y volver a cachear para comparar sin caché
# words_rdd.unpersist()
# repetir las mismas mediciones sin persistencia


## 7. Cierre del ejercicio

En este notebook:

- Creaste un RDD a partir de un archivo de texto.
- Aplicaste transformaciones clave (`flatMap`, `map`, `filter`, `distinct`, `reduceByKey`, `sortBy`).
- Ejecutaste acciones (`count`, `takeSample`, `sum`, `mean`, `stdev`).
- Inspeccionaste el linaje con `toDebugString()` y conectaste esto con *lazy evaluation* y *Jobs*.
- Usaste `persist()` para cachear datos y observar mejoras de tiempo en operaciones repetidas.

Este flujo cubre los criterios del ejercicio de Elementos básicos de Spark (RDDs).